In [ ]:
import os
os.environ["GROQ_API_KEY"]="Enter your api key "

In [ ]:
!pip install -q langchain-groq langchain-core requests

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
@tool
def multiply(a:int, b:int)->int:
  """given two numbers this tools returen product"""
  return a*b

In [ ]:
res=multiply.invoke({"a":5,"b":10})

In [ ]:
print(res)

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)


In [ ]:
llm_with_tools=llm.bind_tools([multiply])

In [ ]:
llm_with_tools

In [ ]:
llm_with_tools.invoke("what is your name?").content

for the above question llm dont need tools to generate the output but for the query that requered to use tools are called by llms by tool binding

Tool calls in an LLM (Large Language Model) are a way for the AI model to use external tools, APIs, functions, or systems to perform tasks beyond just generating text.

In [ ]:
query=HumanMessage("can you multiply 3 with 10")

In [ ]:
message=[query]

In [ ]:
message

In [ ]:
result=llm_with_tools.invoke(message)

In [ ]:
result

In [ ]:
message.append(result)

In [ ]:
message

tool_calls llms called it becaues it required


In [ ]:
result.tool_calls

In [ ]:
tool_result=multiply.invoke(result.tool_calls[0])

In [ ]:
message.append(tool_result)

In [ ]:
message

In [ ]:
llm_with_tools.invoke(message).content

***Currency converter tool ***

In [ ]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [ ]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
  """this function fetches the currency factor between a given base currency and a target currency """

  url=f"https://v6.exchangerate-api.com/v6/ff8031123f5ef086e1142a64/pair/{base_currency}/{target_currency}"
  response=requests.get(url)
  return response.json()

@tool
def convert(base_currency:float,conversion_rate:Annotated[float,InjectedToolArg])->float:
  """given a currency conversion rate this function calculates the target currency value from a given base currency value """
  return base_currency*conversion_rate


In [ ]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

In [ ]:
convert.invoke({'base_currency':10,"conversion_rate":96.6491})

tool binding


In [ ]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [ ]:
messages=[HumanMessage("what is the conversion factor between the USD and INR and based on that convert 10 USD in INR")]

In [ ]:
messages

In [ ]:
ai_message=llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

In [ ]:
import json

In [ ]:
for tool_call in ai_message.tool_calls:
  # execute the first took and get the value of conversion rate
  if tool_call['name']=='get_conversion_factor':
    tool_message1=get_conversion_factor.invoke(tool_call)
    # fetch the conversion rate
    conversion_rate=json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

  if tool_call['name']=='convert':
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message2=convert.invoke(tool_call)
    messages.append(tool_message2)

In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content